In [ ]:
!pip install pyvis

In [ ]:
from pyvis.network import Network
from rdflib import Graph, URIRef
from rdflib.namespace import RDF
import networkx as nx
import sys
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR

In [ ]:
g = Graph()
g.parse(str(DATA_DIR / 'processed' / 'supplychain_kg.ttl'), format="turtle")

# Use remote CDN resources to avoid blocked notebook viewers
net = Network(height="750px", width="100%", notebook=True, cdn_resources='remote')

uri_nodes = set()
edges = []
literals = []

for s, p, o in g:
    p_str = str(p)
    localname = p_str.split('#')[-1] if ('#' in p_str) else p_str.split('/')[-1]
    if isinstance(s, URIRef) and str(s).startswith("http://example.org/supplychain/"):
        s_id = str(s)
        uri_nodes.add(s_id)
        if isinstance(o, URIRef) and str(o).startswith("http://example.org/supplychain/"):
            o_id = str(o)
            uri_nodes.add(o_id)
            edges.append((s_id, o_id, localname))
        else:
            lit_id = f"{s_id}_{localname}_{hash(o)}"
            literals.append((lit_id, s_id, str(o), localname))

G = nx.Graph()
G.add_nodes_from(list(uri_nodes))
for a, b, _label in edges:
    G.add_edge(a, b)

# Limit the visualization to the highest-degree nodes to keep the graph readable
max_uri_nodes = 80
selected_nodes = [n for n, _ in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:max_uri_nodes]]
selected_set = set(selected_nodes)

filtered_edges = [(a, b, lab) for a, b, lab in edges if a in selected_set and b in selected_set]

net.add_node("ROOT", label="Supply Chain Graph", color="#f0a000", shape="star", size=30, title="Top 80 Supply Chain entities by degree")
for node in selected_nodes:
    net.add_node(node, label=node.split("/")[-1], title=node, color="#97C2FC")
    net.add_edge("ROOT", node, color="rgba(100,100,100,0.2)", width=1, dashes=True)

literal_map = {}
for lit_id, parent_id, lit_val, lt in literals:
    if parent_id in selected_set:
        literal_map.setdefault(parent_id, []).append((lit_id, lit_val, lt))

for parent_id, literal_values in literal_map.items():
    for lit_id, lit_val, lt in literal_values[:2]:
        net.add_node(lit_id, label=lit_val, title=lt, shape="box", color="#FFA500")
        net.add_edge(parent_id, lit_id, label=lt, color="#FFA500")

for a, b, lab in filtered_edges:
    net.add_edge(a, b, label=lab)

net.set_options("""
var options = {
  "physics": {
    "enabled": true,
    "stabilization": {
      "enabled": true,
      "iterations": 1200,
      "updateInterval": 50
    },
    "barnesHut": {
      "gravitationalConstant": -8000,
      "centralGravity": 0.3,
      "springLength": 200,
      "springConstant": 0.001,
      "damping": 0.09
    }
  },
  "interaction": {
    "dragNodes": true,
    "dragView": true,
    "hover": true
  },
  "nodes": {
    "font": {
      "size": 14
    },
    "shape": "dot"
  },
  "edges": {
    "smooth": false
  }
}
""")

(DATA_DIR / 'processed').mkdir(parents=True, exist_ok=True)
output_file = str(DATA_DIR / 'processed' / 'supplychain_kg.html')
net.show(output_file)
from IPython.display import HTML, display
with open(output_file, 'r', encoding='utf-8') as f:
    html = f.read()
#display(HTML(html))


/Users/maheshdhiman/GenerativeAI/supply-chain-optimization-kg/supply-chain-optimization-kg/data/processed/supplychain_kg.html
